## Подготовка данных с использованием фреймворка Apache Spark

Подключим необходимые библиотеки.

In [1]:
import os
from pyspark.sql import SparkSession, DataFrame
from pyspark import SparkConf
from pyspark.sql.functions import (
    regexp_replace,
    regexp_extract_all,
    regexp_extract,
    col,
    lit,
    when,
    to_date,
    from_unixtime,
    split,
    trim,
    year,
    udf, 
    array,
    expr
)
from pyspark.ml.feature import VectorAssembler, StringIndexer, OneHotEncoder
from pyspark.sql import functions as F
from pyspark.ml.linalg import Vectors, VectorUDT
from pyspark.ml import Pipeline
from pyspark.sql.types import DoubleType
from pyspark.ml.classification import GBTClassifier, LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.functions import vector_to_array
from pyspark.sql.functions import col, when, array, lit, explode, expr
from sklearn.metrics import roc_curve, auc as sklearn_auc, RocCurveDisplay, accuracy_score, precision_score, recall_score, f1_score
import numpy as np
import matplotlib.pyplot as plt
import time

Сформируем объект конфигурации для `Apache Spark`, указав необходимые параметры.

In [2]:
def create_spark_configuration() -> SparkConf:
    """
    Создает и конфигурирует экземпляр SparkConf для приложения Spark.

    Returns:
        SparkConf: Настроенный экземпляр SparkConf.
    """

    conf = SparkConf()
    conf.setAppName("Load_DP")
    conf.setMaster("local[*]")
    conf.set("spark.driver.memory", "10g")           
    conf.set("spark.executor.memory", "15g")         
    conf.set("spark.memory.fraction", "0.8")        
    conf.set("spark.memory.storageFraction", "0.3") 
    
    # Настройки для больших данных
    conf.set("spark.sql.adaptive.enabled", "true")
    conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")
    conf.set("spark.sql.adaptive.skew.enabled", "true")
    conf.set("spark.sql.shuffle.partitions", "100")  # Уменьшил партиции

    return conf

Создаём сам объект конфигурации.

In [3]:
conf = create_spark_configuration()

Создаём и выводим на экран сессию `Apache Spark`.

In [4]:
spark = SparkSession.builder.config(conf=conf).getOrCreate()
spark

Для исследования будем использовать датасет, расположенный по адресу https://ods.ai/competitions/dl-fintech-bki/data.

Указываем путь.

In [5]:
database_name = "database"
path_features = f"{database_name}/train_features"

train_features = spark.read.parquet(path_features)

Выводим фрагмент датафрейма на экран.

In [6]:
# 1. Статические веса
def get_static_weights(df, label_col='flag'):
    """Простые статические веса"""
    neg_count = df.filter(col(label_col) == 0).count()
    pos_count = df.filter(col(label_col) == 1).count()
    
    # ✅ ПРАВИЛЬНЫЙ вес для балансировки классов
    class_weight = neg_count / pos_count  # 2,893,558 / 106,442 = 27.18
    
    print(f"Статические веса: класс 1 = {class_weight:.2f}, класс 0 = 1.00")
    
    return df.withColumn(
        "final_weight",
        when(col(label_col) == 1, class_weight).otherwise(1.0)
    ).select("id", "features", label_col, "final_weight")

In [7]:
class FastMarginWeighting:
    def calculate_weights(self, df, label_col='flag'):
        # 1. Целевой вес для балансировки
        neg_count = df.filter(col(label_col) == 0).count()
        pos_count = df.filter(col(label_col) == 1).count()
        target_ratio = neg_count / pos_count
        
        # 2. Базовые веса для ОБОИХ классов
        df = df.withColumn(
            "base_weight",
            when(col(label_col) == 1, target_ratio)
            .otherwise(1.0)  # Класс 0 тоже получает вес!
        )
        
        # 3. Вычисляем margin фактор для ВСЕХ объектов
        def get_margin_factor(v):
            arr = v.toArray()
            first_feature = float(arr[0]) if len(arr) > 0 else 0.0
            return first_feature
        
        first_feature_udf = udf(get_margin_factor, DoubleType())
        df = df.withColumn("first_feature", first_feature_udf("features"))
        
        # Нормализуем признак
        stats = df.agg(
            F.mean("first_feature").alias("mean"),
            F.stddev("first_feature").alias("std")
        ).first()
        
        mean_f = stats["mean"]
        std_f = stats["std"] if stats["std"] else 1.0
        
        # 4. Margin вес для ВСЕХ объектов
        df = df.withColumn(
            "z_score", 
            (col("first_feature") - mean_f) / (std_f + 1e-6)
        ).withColumn(
            "margin_factor", 
            F.exp(-F.abs(col("z_score")) / 2.0)
        ).withColumn(
            "margin_weight",
            col("base_weight") * (1.0 + (1.0 - col("margin_factor")))
        )
        
        # 5. ✅ НОРМАЛИЗАЦИЯ для ВСЕХ объектов (ОБА класса)
        mean_weight = df.agg(F.mean("margin_weight")).first()[0]
        scale_factor = 1.0 / mean_weight  # Нормализуем к среднему 1.0
        
        result = df.withColumn(
            "final_weight",
            col("margin_weight") * scale_factor
        )
        
        # 6. Статистика по классам
        stats_by_class = result.groupBy(label_col).agg(
            F.mean("final_weight").alias("mean"),
            F.stddev("final_weight").alias("std"),
            F.min("final_weight").alias("min"),
            F.max("final_weight").alias("max")
        ).collect()
        
        for row in stats_by_class:
            cls = row[label_col]
            print(f"  Margin класс {cls}: μ={row['mean']:.2f}±{row['std']:.2f} "
                  f"[{row['min']:.2f}, {row['max']:.2f}]")
        
        return result.select("id", "features", label_col, "final_weight")

In [8]:
class FastDensityWeighting:
    def calculate_weights(self, df, label_col='flag'):
        # 1. Целевое соотношение
        neg_count = df.filter(col(label_col) == 0).count()
        pos_count = df.filter(col(label_col) == 1).count()
        target_ratio = neg_count / pos_count
        
        # 2. Базовые веса для ОБОИХ классов
        df = df.withColumn(
            "base_weight",
            when(col(label_col) == 1, target_ratio)
            .otherwise(1.0)
        )
        
        # 3. Кластеризация для ВСЕХ объектов
        from pyspark.ml.clustering import KMeans
        from pyspark.ml.feature import StandardScaler
        
        # Масштабируем
        scaler = StandardScaler(inputCol="features", outputCol="scaled_features",
                                withStd=True, withMean=True)
        
        sample_df = df.sample(fraction=0.1, seed=42).cache()
        scaler_model = scaler.fit(sample_df)
        
        # Оптимальное k
        k = self._find_optimal_k(scaler_model.transform(sample_df))
        
        # KMeans
        kmeans = KMeans(k=k, seed=42, featuresCol="scaled_features", maxIter=20)
        kmeans_model = kmeans.fit(scaler_model.transform(sample_df))
        
        # Применяем ко ВСЕМ данным
        df_scaled = scaler_model.transform(df)
        clustered = kmeans_model.transform(df_scaled)
        
        # Размер кластеров
        cluster_counts = clustered.groupBy("prediction").count()
        clustered = clustered.join(cluster_counts, "prediction")
        
        total_count = df.count()
        
        # 4. Веса для ВСЕХ объектов на основе плотности
        df = clustered.withColumn(
            "cluster_freq", 
            col("count") / total_count
        ).withColumn(
            "density_factor", 
            1.0 / (F.sqrt(col("cluster_freq")) + 0.01)  # Квадратный корень для сглаживания
        ).withColumn(
            "density_weight",
            col("base_weight") * col("density_factor")
        )
        
        sample_df.unpersist()
        
        # 5. ✅ НОРМАЛИЗАЦИЯ для ВСЕХ объектов
        mean_weight = df.agg(F.mean("density_weight")).first()[0]
        scale_factor = 1.0 / mean_weight
        
        result = df.withColumn(
            "final_weight",
            col("density_weight") * scale_factor
        )
        
        # 6. Статистика
        stats_by_class = result.groupBy(label_col).agg(
            F.mean("final_weight").alias("mean"),
            F.stddev("final_weight").alias("std"),
            F.min("final_weight").alias("min"),
            F.max("final_weight").alias("max")
        ).collect()
        
        for row in stats_by_class:
            cls = row[label_col]
            print(f"  Density класс {cls}: μ={row['mean']:.2f}±{row['std']:.2f} "
                  f"[{row['min']:.2f}, {row['max']:.2f}]")
        
        return result.select("id", "features", label_col, "final_weight")
    
    def _find_optimal_k(self, df, max_k=20):
        """Автоматический выбор k"""
        from pyspark.ml.clustering import KMeans
        import numpy as np
        
        ks = range(2, max_k, 2)
        costs = []
        
        for k in ks:
            kmeans = KMeans(k=k, seed=42, maxIter=10)
            model = kmeans.fit(df)
            costs.append(model.summary.trainingCost)
        
        # Elbow method
        diffs = np.diff(costs)
        diff_rates = diffs[:-1] / (diffs[1:] + 1e-6)
        optimal_idx = np.argmax(diff_rates) if len(diff_rates) > 0 else 3
        
        return ks[min(optimal_idx, len(ks)-1)]

In [9]:
class FocalLossWeighting:
    def __init__(self, gamma=2.0, alpha=0.25):
        self.gamma = gamma
        self.alpha = alpha
    
    def calculate_weights(self, df, label_col='flag'):
        # 1. Базовое соотношение
        neg_count = df.filter(col(label_col) == 0).count()
        pos_count = df.filter(col(label_col) == 1).count()
        base_weight_pos = neg_count / pos_count
        
        # 2. Базовые веса для ОБОИХ классов
        df = df.withColumn(
            "base_weight",
            when(col(label_col) == 1, base_weight_pos)
            .otherwise(1.0)
        )
        
        # 3. Быстрая логистическая регрессия
        from pyspark.ml.classification import LogisticRegression
        
        sample_df = df.sample(fraction=0.1, seed=42).cache()
        lr = LogisticRegression(
            featuresCol="features", 
            labelCol=label_col, 
            maxIter=10, 
            regParam=0.1
        )
        model = lr.fit(sample_df)
        predictions = model.transform(df)
        
        # 4. Focal фактор для ВСЕХ объектов
        def focal_weight(prob, label):
            p = float(prob[1]) if prob else 0.5
            # Focal Loss формула для весов
            pt = p if label == 1 else 1 - p
            focal = (1 - pt) ** self.gamma
            return float(focal)
        
        focal_udf = udf(focal_weight, DoubleType())
        
        result = predictions.withColumn(
            "focal_factor",
            focal_udf(col("probability"), col(label_col))
        ).withColumn(
            "focal_weight",
            col("base_weight") * (1.0 + self.alpha * col("focal_factor"))
        )
        
        sample_df.unpersist()
        
        # 5. ✅ НОРМАЛИЗАЦИЯ для ВСЕХ объектов
        mean_weight = result.agg(F.mean("focal_weight")).first()[0]
        scale_factor = 1.0 / mean_weight
        
        result = result.withColumn(
            "final_weight",
            col("focal_weight") * scale_factor
        )
        
        # 6. Статистика
        stats_by_class = result.groupBy(label_col).agg(
            F.mean("final_weight").alias("mean"),
            F.stddev("final_weight").alias("std")
        ).collect()
        
        for row in stats_by_class:
            cls = row[label_col]
            print(f"  Focal класс {cls}: μ={row['mean']:.3f}±{row['std']:.3f}")
        
        return result.select("id", "features", label_col, "final_weight")

In [10]:
class DifficultyAwareWeighting:
    def calculate_weights(self, df, label_col='flag'):
        # 1. Целевое соотношение
        neg_count = df.filter(col(label_col) == 0).count()
        pos_count = df.filter(col(label_col) == 1).count()
        target_ratio = neg_count / pos_count
        
        # 2. GBT для оценки сложности
        from pyspark.ml.classification import GBTClassifier
        
        sample_df = df.sample(fraction=0.2, seed=42).cache()
        
        gbt = GBTClassifier(
            featuresCol="features",
            labelCol=label_col,
            maxIter=10,
            maxDepth=4,
            seed=42
        )
        
        model = gbt.fit(sample_df)
        predictions = model.transform(df)
        
        # 3. Сложность для ВСЕХ объектов
        def compute_difficulty(prob):
            p = float(prob[1]) if prob else 0.5
            # Более плавная функция сложности
            difficulty = 1.0 - 2.0 * (p - 0.5) ** 2  # Параболическая
            return float(difficulty)
        
        difficulty_udf = udf(compute_difficulty, DoubleType())
        
        df = predictions.withColumn(
            "difficulty",
            difficulty_udf(col("probability"))
        ).withColumn(
            "base_weight",
            when(col(label_col) == 1, target_ratio)
            .otherwise(1.0)
        ).withColumn(
            "difficulty_weight",
            col("base_weight") * (1.0 + col("difficulty"))
        )
        
        sample_df.unpersist()
        
        # 4. ✅ НОРМАЛИЗАЦИЯ для ВСЕХ объектов
        mean_weight = df.agg(F.mean("difficulty_weight")).first()[0]
        scale_factor = 1.0 / mean_weight
        
        result = df.withColumn(
            "final_weight",
            col("difficulty_weight") * scale_factor
        )
        
        # 5. Статистика
        stats_by_class = result.groupBy(label_col).agg(
            F.mean("final_weight").alias("mean"),
            F.stddev("final_weight").alias("std")
        ).collect()
        
        for row in stats_by_class:
            cls = row[label_col]
            print(f"  Difficulty класс {cls}: μ={row['mean']:.2f}±{row['std']:.2f}")
        
        return result.select("id", "features", label_col, "final_weight")

In [11]:
from pyspark.sql.functions import col, when, lit, sqrt, sum as spark_sum
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.linalg import Vectors, VectorUDT
from pyspark.sql.types import DoubleType
import numpy as np
import time

class HierarchicalAdaptiveWeighting:
    """
    МАКСИМАЛЬНО БЫСТРАЯ ВЕРСИЯ - ТОЛЬКО БАЛАНС + ПРОСТАЯ СЛОЖНОСТЬ
    Время: ~30 секунд на 3M объектов
    """
    def __init__(self, alpha=0.7, beta=0.3):
        self.alpha = alpha
        self.beta = beta
    
    def calculate_weights(self, df, label_col='flag'):
        print("="*70)
        print("АДАПТИВНОЕ ВЗВЕШИВАНИЕ (МИНИМАЛЬНОЕ)")
        print("="*70)
        
        start = time.time()
        
        # 1. Балансировка классов
        neg_count = df.filter(col(label_col) == 0).count()
        pos_count = df.filter(col(label_col) == 1).count()
        target_weight = neg_count / pos_count
        
        # 2. Простые веса
        df = df.withColumn(
            "balance_weight",
            when(col(label_col) == 1, target_weight).otherwise(1.0)
        )
        
        # 3. Очень простая сложность через UDF
        def simple_norm(v):
            if v is None:
                return 1.0
            try:
                arr = v.toArray()
                norm = np.sqrt(np.sum(arr ** 2))
                # Мягкое экспоненциальное масштабирование
                return float(1.0 + np.tanh(norm / 1000))
            except:
                return 1.0
        
        norm_udf = udf(simple_norm, DoubleType())
        
        df = df.withColumn(
            "difficulty_factor",
            norm_udf(col("features"))
        )
        
        # 4. Комбинируем
        df = df.withColumn(
            "final_weight",
            self.alpha * col("balance_weight") + 
            self.beta * col("difficulty_factor") * col("balance_weight")
        )
        
        # 5. Нормализация
        mean_w = df.agg(F.mean("final_weight")).first()[0]
        df = df.withColumn(
            "final_weight",
            col("final_weight") / mean_w
        )
        
        # 6. Статистика
        w1 = df.filter(col(label_col) == 1).agg(F.mean("final_weight")).first()[0]
        w0 = df.filter(col(label_col) == 0).agg(F.mean("final_weight")).first()[0]
        
        print(f"\n  Время: {time.time() - start:.1f} сек")
        print(f"  Соотношение весов: {w1/w0:.3f}")
        print(f"  Класс 0: μ={w0:.3f}")
        print(f"  Класс 1: μ={w1:.3f}")
        
        return df.select("id", "features", label_col, "final_weight")

In [12]:
def weighting_comparison(train_df, label_col='flag'):
    """Сравнение методов взвешивания"""
    
    print("="*60)
    print("СРАВНЕНИЕ МЕТОДОВ ВЗВЕШИВАНИЯ")
    print("="*60)
    
    # 1. Статические веса
    static_df = get_static_weights(train_df, label_col)
    
    # 2. Fast Margin-Based
    mbw = FastMarginWeighting()
    margin_df = mbw.calculate_weights(train_df, label_col)
    
    # 3. Fast Density-Based
    dbw = FastDensityWeighting()
    density_df = dbw.calculate_weights(train_df, label_col)
    
    # 4. Focal Loss
    flw = FocalLossWeighting(gamma=2.0, alpha=0.25)
    focal_df = flw.calculate_weights(train_df, label_col)
    
    # 5. Difficulty Aware
    daw = DifficultyAwareWeighting()
    difficulty_df = daw.calculate_weights(train_df, label_col)
    
    # 6. HierarchicalAdaptive (с оптимальными коэффициентами)
    cw = HierarchicalAdaptiveWeighting()
    hierarchicalAdaptive_df = cw.calculate_weights(train_df, label_col)
    
    # Вывод статистики
    print("\n" + "="*60)
    print("СТАТИСТИКА ВЕСОВ")
    print("="*60)
    
    for name, df in [
        ("Статический", static_df),
        ("Margin-Based", margin_df),
        ("Density-Based", density_df),
        ("Focal Loss", focal_df),
        ("Difficulty Aware", difficulty_df),
        ("HierarchicalAdaptive", hierarchicalAdaptive_df)
    ]:
        stats_class1 = df.filter(col(label_col) == 1).agg(
            F.mean("final_weight").alias("mean"),
            F.stddev("final_weight").alias("std"),
            F.min("final_weight").alias("min"),
            F.max("final_weight").alias("max"),
            F.count("*").alias("count")
        ).collect()[0]
        
        stats_class0 = df.filter(col(label_col) == 0).agg(
            F.mean("final_weight").alias("mean"),
            F.stddev("final_weight").alias("std"),
            F.count("*").alias("count")
        ).collect()[0]
        
        print(f"\n{name}:")
        print(f"  Класс 1: вес={stats_class1['mean']:.3f}±{stats_class1['std']:.3f} "
              f"[{stats_class1['min']:.3f}, {stats_class1['max']:.3f}], n={stats_class1['count']:,}")
        print(f"  Класс 0: вес={stats_class0['mean']:.3f}±{stats_class0['std']:.3f}, n={stats_class0['count']:,}")
    
    return static_df, margin_df, density_df, focal_df, difficulty_df, hierarchicalAdaptive_df

In [13]:
static_weights, margin_weights, density_weights, focal_weights, difficulty_weights, hierarchicalAdaptive_weights = weighting_comparison(train_features, label_col='flag')

СРАВНЕНИЕ МЕТОДОВ ВЗВЕШИВАНИЯ
Статические веса: класс 1 = 27.18, класс 0 = 1.00
  Margin класс 1: μ=14.19±1.85 [10.77, 20.78]
  Margin класс 0: μ=0.51±0.07 [0.40, 0.77]
  Density класс 1: μ=14.03±2.06 [12.60, 17.01]
  Density класс 0: μ=0.52±0.08 [0.46, 0.63]
  Focal класс 1: μ=15.530±0.140
  Focal класс 0: μ=0.466±0.000
  Difficulty класс 1: μ=14.15±0.24
  Difficulty класс 0: μ=0.52±0.01
АДАПТИВНОЕ ВЗВЕШИВАНИЕ (МИНИМАЛЬНОЕ)

  Время: 101.0 сек
  Соотношение весов: 27.173
  Класс 0: μ=0.519
  Класс 1: μ=14.089

СТАТИСТИКА ВЕСОВ

Статический:
  Класс 1: вес=27.184±0.000 [27.184, 27.184], n=106,442
  Класс 0: вес=1.000±0.000, n=2,893,558

Margin-Based:
  Класс 1: вес=14.186±1.853 [10.768, 20.779], n=106,442
  Класс 0: вес=0.515±0.070, n=2,893,558

Density-Based:
  Класс 1: вес=14.029±2.064 [12.598, 17.006], n=106,442
  Класс 0: вес=0.521±0.078, n=2,893,558

Focal Loss:
  Класс 1: вес=15.530±0.140 [12.786, 15.789], n=106,442
  Класс 0: вес=0.466±0.000, n=2,893,558

Difficulty Aware:
  Кла

In [14]:
static_weights = static_weights.drop("classWeight") \
    .withColumnRenamed("final_weight", "classWeight")

margin_weights = margin_weights.drop("classWeight") \
    .withColumnRenamed("final_weight", "classWeight")

density_weights = density_weights.drop("classWeight") \
    .withColumnRenamed("final_weight", "classWeight")

focal_weights = focal_weights.drop("classWeight") \
    .withColumnRenamed("final_weight", "classWeight")

difficulty_weights = difficulty_weights.drop("classWeight") \
    .withColumnRenamed("final_weight", "classWeight")

hierarchicalAdaptive_weights = hierarchicalAdaptive_weights.drop("classWeight") \
    .withColumnRenamed("final_weight", "classWeight")

#### Обучение модели

In [15]:
def simple_robust_train_and_evaluate(train_data_with_weights, valid_data, method_name="Method", max_samples=50000, model_type="gbt"):
    """
    Простая и устойчивая функция для обучения и оценки
    
    Args:
        train_data_with_weights: DataFrame с колонками features, flag, classWeight
        valid_data: валидационный DataFrame (без весов, только features, flag)
        method_name: название метода для вывода
        max_samples: максимальное количество примеров для обучения
        model_type: тип модели ("gbt" или "logreg")
    """
    
    print(f"\n▶ {method_name}")
    results = {'method': method_name}
    
    try:
        # 1. Проверяем наличие необходимых колонок
        required_cols = ['features', 'flag', 'classWeight']
        for col_name in required_cols:
            if col_name not in train_data_with_weights.columns:
                # Ищем альтернативные названия
                if col_name == 'classWeight':
                    # Пробуем найти колонку с весом
                    weight_cols = [c for c in train_data_with_weights.columns 
                                 if 'weight' in c.lower() or 'final' in c.lower()]
                    if weight_cols:
                        train_data_with_weights = train_data_with_weights.withColumnRenamed(
                            weight_cols[0], "classWeight"
                        )
                        print(f"  Переименовал {weight_cols[0]} → classWeight")
                    else:
                        raise ValueError(f"Колонка с весами не найдена. Доступные колонки: {train_data_with_weights.columns}")
                else:
                    raise ValueError(f"Колонка {col_name} не найдена")
        
        # 2. Опционально: ограничиваем размер данных для скорости
        total_count = train_data_with_weights.count()
        if total_count > max_samples:
            sample_fraction = max_samples / total_count
            train_df = train_data_with_weights.sample(fraction=sample_fraction, seed=42)
            print(f"  Выборка: {sample_fraction:.1%} ({max_samples:,} примеров)")
        else:
            train_df = train_data_with_weights
            print(f"  Используем все данные: {total_count:,} примеров")
        
        # 3. Кэшируем тренировочные данные для скорости
        train_df = train_df.cache()
        
        # 4. Инициализируем модель
        if model_type == "gbt":
            model_estimator = GBTClassifier(
                featuresCol="features",
                labelCol="flag",
                weightCol="classWeight",
                maxIter=20,
                maxDepth=3,           # Увеличил глубину для лучшего качества
                stepSize=0.1,
                subsamplingRate=0.7,
                minInstancesPerNode=1,
                minInfoGain=0.0,
                maxBins=32,
                seed=42,
                cacheNodeIds=False,
                checkpointInterval=10,
                impurity="variance",  # variance для GBT
                featureSubsetStrategy="auto"
            )
            
        elif model_type == "logreg":
            model_estimator = LogisticRegression(
                featuresCol="features",
                labelCol="flag",
                weightCol="classWeight",
                maxIter=30,
                regParam=0.1,
                elasticNetParam=0.0,
                standardization=False,
                threshold=0.5
            )
        
        # 5. Обучаем модель на ДАННЫХ С ВЕСАМИ
        start_time = time.time()
        model = model_estimator.fit(train_df)
        train_time = time.time() - start_time
        
        # 6. Предсказания на валидации (без весов)
        predictions = model.transform(valid_data)
        predictions = predictions.cache()
        
        # 7. Вычисляем метрики
        # AUC-ROC
        evaluator_roc = BinaryClassificationEvaluator(
            labelCol="flag",
            rawPredictionCol="rawPrediction",
            metricName="areaUnderROC"
        )
        auc_score = evaluator_roc.evaluate(predictions)
        
        # PR-AUC
        evaluator_pr = BinaryClassificationEvaluator(
            labelCol="flag",
            rawPredictionCol="rawPrediction",
            metricName="areaUnderPR"
        )
        pr_auc = evaluator_pr.evaluate(predictions)
        
        # Accuracy
        accuracy_df = predictions.withColumn(
            "correct",
            when(col("prediction") == col("flag"), 1).otherwise(0)
        )
        accuracy = accuracy_df.agg(F.avg("correct")).collect()[0][0]
        
        # Precision, Recall, F1
        tp = predictions.filter((col("prediction") == 1) & (col("flag") == 1)).count()
        fp = predictions.filter((col("prediction") == 1) & (col("flag") == 0)).count()
        fn = predictions.filter((col("prediction") == 0) & (col("flag") == 1)).count()
        tn = predictions.filter((col("prediction") == 0) & (col("flag") == 0)).count()
        
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
        specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
        
        # 8. Статистика обучения
        class_counts_train = train_df.groupBy("flag").count().collect()
        train_class_dist = {row['flag']: row['count'] for row in class_counts_train}
        
        class_counts_valid = valid_data.groupBy("flag").count().collect()
        valid_class_dist = {row['flag']: row['count'] for row in class_counts_valid}
        
        # 9. Статистика весов
        weight_stats = train_df.agg(
            F.min("classWeight").alias("min_weight"),
            F.max("classWeight").alias("max_weight"),
            F.mean("classWeight").alias("mean_weight"),
            F.stddev("classWeight").alias("std_weight")
        ).collect()[0]
        
        # 10. Вывод результатов
        print(f"  {'='*40}")
        print(f"  РЕЗУЛЬТАТЫ {method_name}:")
        print(f"  {'='*40}")
        print(f"  ROC-AUC:  {auc_score:.4f}")
        print(f"  PR-AUC:   {pr_auc:.4f}")
        print(f"  Accuracy: {accuracy:.4f}")
        print(f"  Precision:{precision:.4f}")
        print(f"  Recall:   {recall:.4f}")
        print(f"  F1-score: {f1:.4f}")
        print(f"  Specificity: {specificity:.4f}")
        print(f"  {'='*40}")
        print(f"  Время обучения: {train_time:.1f}с")
        print(f"  Распределение классов (train): {train_class_dist}")
        print(f"  Распределение классов (valid): {valid_class_dist}")
        print(f"  Статистика весов:")
        print(f"    min: {weight_stats['min_weight']:.3f}")
        print(f"    max: {weight_stats['max_weight']:.3f}")
        print(f"    mean: {weight_stats['mean_weight']:.3f}")
        print(f"    std: {weight_stats['std_weight']:.3f}")
        
        # 11. Очистка кэша
        train_df.unpersist()
        predictions.unpersist()
        
        # 12. Возвращаем результаты
        return {
            'method': method_name,
            'model_type': model_type,
            'auc': float(auc_score),
            'pr_auc': float(pr_auc),
            'accuracy': float(accuracy),
            'precision': float(precision),
            'recall': float(recall),
            'f1': float(f1),
            'specificity': float(specificity),
            'train_time': float(train_time),
            'train_samples': int(train_df.count()),
            'train_class_dist': train_class_dist,
            'valid_class_dist': valid_class_dist,
            'weight_stats': {
                'min': float(weight_stats['min_weight']),
                'max': float(weight_stats['max_weight']),
                'mean': float(weight_stats['mean_weight']),
                'std': float(weight_stats['std_weight'])
            },
            'status': 'success'
        }, model
        
    except Exception as e:
        print(f"  ✗ ОШИБКА в {method_name}: {str(e)}")
        import traceback
        traceback.print_exc()
        return {
            'method': method_name,
            'model_type': model_type,
            'error': str(e),
            'status': 'failed'
        }, None

In [16]:
train_df, valid_df = train_features.randomSplit([0.8, 0.2], seed=42)

def compare_weighting_methods_with_details(methods_dict, max_samples_per_method=500000):
    """
    Детальное сравнение методов взвешивания
    
    Args:
        methods_dict: словарь {method_name: DataFrame_with_weights}
        valid_df: валидационный DataFrame (без весов)
        max_samples_per_method: максимальное количество примеров для обучения
    """
    
    print("=" * 80)
    print("СРАВНЕНИЕ МЕТОДОВ ВЗВЕШИВАНИЯ (GBT vs Logistic Regression)")
    print("=" * 80)
    
    all_results = {}
    models = {}
    
    # Кэшируем валидационные данные
    valid_df_cached = valid_df.cache()
    valid_count = valid_df_cached.count()
    print(f"\nВалидационная выборка: {valid_count:,} примеров")
    
    for method_name, train_data in methods_dict.items():
        print(f"\n{'=' * 50}")
        print(f"МЕТОД ВЗВЕШИВАНИЯ: {method_name}")
        print(f"{'=' * 50}")
        
        # Проверяем наличие данных
        if train_data is None:
            print(f"✗ Нет данных для метода {method_name}")
            continue
        
        # Проверяем размер
        train_count = train_data.count()
        print(f"Размер обучающей выборки: {train_count:,} примеров")
        
        # ==============================
        # 1. GBT
        # ==============================
        result_gbt, model_gbt = simple_robust_train_and_evaluate(
            train_data_with_weights=train_data,
            valid_data=valid_df_cached,
            method_name=f"{method_name}_GBT",
            max_samples=max_samples_per_method,
            model_type="gbt"
        )
        
        if result_gbt["status"] == "success":
            all_results[f"{method_name}_GBT"] = result_gbt
            models[f"{method_name}_GBT"] = model_gbt
        
        # ==============================
        # 2. Logistic Regression
        # ==============================
        result_lr, model_lr = simple_robust_train_and_evaluate(
            train_data_with_weights=train_data,
            valid_data=valid_df_cached,
            method_name=f"{method_name}_LR",
            max_samples=max_samples_per_method,
            model_type="logreg"
        )
        
        if result_lr["status"] == "success":
            all_results[f"{method_name}_LR"] = result_lr
            models[f"{method_name}_LR"] = model_lr
    
    # Очищаем кэш валидации
    valid_df_cached.unpersist()
    
    # ==============================
    # Итоговая таблица
    # ==============================
    successful = {k: v for k, v in all_results.items() 
                 if v.get("status") == "success"}
    
    if successful:
        print("\n" + "=" * 120)
        print("ИТОГОВАЯ ТАБЛИЦА (СОРТИРОВКА ПО PR-AUC)")
        print("=" * 120)
        
        print(f"{'Метод':<25} {'ROC-AUC':>8} {'PR-AUC':>8} {'F1':>8} "
              f"{'Prec':>8} {'Recall':>8} {'Время':>8} {'Вес(сред)':>10}")
        print("-" * 120)
        
        # Сортируем по PR-AUC
        for method, stats in sorted(successful.items(),
                                   key=lambda x: x[1].get('pr_auc', 0),
                                   reverse=True):
            
            weight_mean = stats.get('weight_stats', {}).get('mean', 0)
            
            print(f"{method:<25} "
                  f"{stats.get('auc', 0):>8.4f} "
                  f"{stats.get('pr_auc', 0):>8.4f} "
                  f"{stats.get('f1', 0):>8.4f} "
                  f"{stats.get('precision', 0):>8.4f} "
                  f"{stats.get('recall', 0):>8.4f} "
                  f"{stats.get('train_time', 0):>7.1f}с "
                  f"{weight_mean:>10.3f}")
        
        # Добавляем baseline без весов для сравнения
        print("\n" + "-" * 120)
        
    return all_results, models

In [17]:
def prepare_comparison_data():
    """
    Подготовка данных для сравнения
    """
    methods = {
        'Static': static_weights,
        'Margin': margin_weights,
        'Density': density_weights,
        'Focal': focal_weights,
        'Difficulty': difficulty_weights,
        'HierarchicalAdaptive': hierarchicalAdaptive_weights
    }
    return methods

In [18]:
methods = prepare_comparison_data()
all_results = compare_weighting_methods_with_details(methods, max_samples_per_method=500000)

СРАВНЕНИЕ МЕТОДОВ ВЗВЕШИВАНИЯ (GBT vs Logistic Regression)

Валидационная выборка: 599,277 примеров

МЕТОД ВЗВЕШИВАНИЯ: Static
Размер обучающей выборки: 3,000,000 примеров

▶ Static_GBT
  Выборка: 16.7% (500,000 примеров)
  РЕЗУЛЬТАТЫ Static_GBT:
  ROC-AUC:  0.6979
  PR-AUC:   0.0807
  Accuracy: 0.6499
  Precision:0.0628
  Recall:   0.6306
  F1-score: 0.1143
  Specificity: 0.6507
  Время обучения: 20.9с
  Распределение классов (train): {1: 17776, 0: 482775}
  Распределение классов (valid): {1: 21456, 0: 577821}
  Статистика весов:
    min: 1.000
    max: 27.184
    mean: 1.930
    std: 4.846

▶ Static_LR
  Выборка: 16.7% (500,000 примеров)
  РЕЗУЛЬТАТЫ Static_LR:
  ROC-AUC:  0.6714
  PR-AUC:   0.0665
  Accuracy: 0.6285
  Precision:0.0596
  Recall:   0.6341
  F1-score: 0.1089
  Specificity: 0.6283
  Время обучения: 12.3с
  Распределение классов (train): {1: 17776, 0: 482775}
  Распределение классов (valid): {1: 21456, 0: 577821}
  Статистика весов:
    min: 1.000
    max: 27.184
    mea

После успешной записи таблицы останавливаем сессию `Apache Spark`.

In [19]:
spark.stop()